In [ ]:
from pathlib import Path
import numpy as np
import tifffile
import traceback

# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
PATCH_ROOT = Path(r"D:\Sunita_Thesis\Datasets\Data_Patches_1km_DEMCropped")

# Output folder for merged 2 km patches as .npy
OUT_2KM = Path(r"D:\Sunita_Thesis\Datasets\Data_Patches_2km_noDEM")
OUT_2KM.mkdir(parents=True, exist_ok=True)

EXPECTED_TILE_SHAPE = (100, 100)

# ✅ CHANGED: only 5 channels, no DEM
CHANNELS = ["R", "G", "B", "NIR", "NOISE"]


# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def parse_tile_index(tile_id: str):
    try:
        part = tile_id.split("_")[2]  # '2501-1120'
        col_str, row_str = part.split("-")
        return int(col_str), int(row_str)
    except Exception as e:
        raise ValueError(f"Could not parse (col,row) from tile_id '{tile_id}': {e}")


def get_ul_world(dem_path: Path):
    with tifffile.TiffFile(dem_path) as tf:
        page = tf.pages[0]
        tags = page.tags
        scale = tags.get(33550)  # ModelPixelScaleTag
        tie   = tags.get(33922)  # ModelTiepointTag

        if scale is None or tie is None:
            raise RuntimeError(f"Missing GeoTIFF tags in {dem_path}")

        sx, sy = float(scale.value[0]), float(scale.value[1])
        i0, j0, _, X0, Y0, _ = map(float, tie.value[:6])

        ulx = X0 + i0 * sx
        uly = Y0 - j0 * sy
    return ulx, uly


# ✅ CHANGED: loads only 5 channels now
def load_tile_five_channels(folder: Path, tile_id: str):
    """
    Load the 5 channels (R, G, B, NIR, NOISE) for a 1 km tile.
    Returns array of shape (5, H, W).
    """
    arrays = []
    shapes = []

    for ch in CHANNELS:
        path = folder / f"{tile_id}__{ch}.tif"
        if not path.exists():
            raise FileNotFoundError(f"Missing {ch} channel for tile {tile_id}: {path}")
        arr = tifffile.imread(path)
        arrays.append(arr)
        shapes.append(arr.shape)

    first_shape = shapes[0]
    if not all(s == first_shape for s in shapes):
        raise ValueError(f"Channel shape mismatch in tile {tile_id}: {shapes}")

    if first_shape != EXPECTED_TILE_SHAPE:
        raise ValueError(f"Tile {tile_id}: shape={first_shape}, expected={EXPECTED_TILE_SHAPE}")

    return np.stack(arrays, axis=0)  # (5, H, W)


# -------------------------------------------------------------------
# 1. SCAN ALL 1 KM TILES AND BUILD INDEX GRID
# -------------------------------------------------------------------
tile_dict = {}   # key=(col,row) -> {"id": tile_id, "folder": folder}

print("Scanning 1 km tile folders...")

for folder in sorted(PATCH_ROOT.iterdir()):
    if not folder.is_dir():
        continue

    tile_id = folder.name

    # ✅ Keep DEM check ONLY for: (1) validity & (2) coordinate tags
    dem_path = folder / f"{tile_id}__DEM.tif"
    if not dem_path.exists():
        continue

    try:
        col, row = parse_tile_index(tile_id)
    except ValueError as e:
        print(f"  ⚠ Skipping folder {folder.name}: {e}")
        continue

    tile_dict[(col, row)] = {
        "id": tile_id,
        "folder": folder,
        "dem_path": dem_path,   # used only for UL coords
    }

print(f"Found {len(tile_dict)} valid 1 km tiles.")

if not tile_dict:
    raise RuntimeError("No valid tiles found. Check PATCH_ROOT and naming conventions.")


all_cols = sorted({c for (c, r) in tile_dict.keys()})
all_rows = sorted({r for (c, r) in tile_dict.keys()}, reverse=True)

print(f"Columns range: {all_cols[0]} .. {all_cols[-1]}  (count {len(all_cols)})")
print(f"Rows range   : {all_rows[-1]} .. {all_rows[0]}  (count {len(all_rows)})")


# -------------------------------------------------------------------
# 2. MERGE 2×2 BLOCKS INTO (5, 200, 200) PATCHES WITH RESUME SUPPORT
# -------------------------------------------------------------------
error_log = []
merged_count = 0
skipped_blocks_missing = 0
block_idx = 1

for i in range(len(all_rows) - 1):
    row_top = all_rows[i]
    row_bot = all_rows[i + 1]

    for j in range(len(all_cols) - 1):
        col_left  = all_cols[j]
        col_right = all_cols[j + 1]

        key_NW = (col_left,  row_top)
        key_NE = (col_right, row_top)
        key_SW = (col_left,  row_bot)
        key_SE = (col_right, row_bot)

        if not all(k in tile_dict for k in [key_NW, key_NE, key_SW, key_SE]):
            skipped_blocks_missing += 1
            block_idx += 1
            continue

        nw_tile = tile_dict[key_NW]
        nw_id = nw_tile["id"]
        nw_dem_path = nw_tile["dem_path"]

        try:
            ulx, uly = get_ul_world(nw_dem_path)
        except Exception as e:
            print(f"  ⚠ Could not read UL coords for {nw_id}: {e}")
            ulx, uly = 0.0, 0.0

        # ✅ Naming stays the same => still unique + real-world loc traceable
        fname = f"{block_idx:04d}_col{col_left}_row{row_top}_x{int(round(ulx))}_y{int(round(uly))}.npy"
        out_path = OUT_2KM / fname

        if out_path.exists():
            print(f"⏩ Block {block_idx:04d} already exists, skipping ({fname})")
            block_idx += 1
            continue

        try:
            # ✅ CHANGED: load 5-channel tiles
            A_NW = load_tile_five_channels(tile_dict[key_NW]["folder"], tile_dict[key_NW]["id"])
            A_NE = load_tile_five_channels(tile_dict[key_NE]["folder"], tile_dict[key_NE]["id"])
            A_SW = load_tile_five_channels(tile_dict[key_SW]["folder"], tile_dict[key_SW]["id"])
            A_SE = load_tile_five_channels(tile_dict[key_SE]["folder"], tile_dict[key_SE]["id"])

            top = np.concatenate([A_NW, A_NE], axis=2)       # (5,100,200)
            bottom = np.concatenate([A_SW, A_SE], axis=2)   # (5,100,200)
            merged = np.concatenate([top, bottom], axis=1)  # (5,200,200)

            np.save(out_path, merged)
            merged_count += 1
            print(f"✓ Block {block_idx:04d} -> {fname}")

        except Exception as e:
            print(f"❌ Error in block {block_idx:04d} (col {col_left}-{col_right}, row {row_top}-{row_bot}): {e}")
            tb = traceback.format_exc()
            error_log.append({
                "block_idx": block_idx,
                "cols": (col_left, col_right),
                "rows": (row_top, row_bot),
                "error": str(e),
                "traceback": tb,
            })

        block_idx += 1

print("\n---------------- SUMMARY ----------------")
print(f"Merged blocks saved        : {merged_count}")
print(f"Skipped (missing neighbours): {skipped_blocks_missing}")
print(f"Blocks with errors         : {len(error_log)}")

if error_log:
    log_path = OUT_2KM / "merge_errors.txt"
    with open(log_path, "w", encoding="utf-8") as f:
        for e in error_log:
            f.write(f"Block {e['block_idx']} cols={e['cols']} rows={e['rows']}\n")
            f.write(e["traceback"])
            f.write("\n" + "-"*40 + "\n")
    print(f"Error details written to: {log_path}")


In [ ]:
from pathlib import Path
import csv

PATCH_DIR = Path(r"D:\Sunita_Thesis\Datasets\Data_Patches_2km_noDEM")

def parse_patch_name(path: Path):
    """
    Parse patch filename of the form:
      0235_col2719_row1297_x502150_y5296630.npy

    Returns:
      patch_idx, col, row, ulx, uly
    """
    stem = path.stem
    parts = stem.split("_")

    # ✅ SAFETY CHECK: strict filename format
    if len(parts) != 5:
        raise ValueError(f"Unexpected patch filename format: {path.name}")

    patch_idx = int(parts[0])
    col = int(parts[1].replace("col", ""))
    row = int(parts[2].replace("row", ""))
    ulx = int(parts[3].replace("x", ""))
    uly = int(parts[4].replace("y", ""))

    return patch_idx, col, row, ulx, uly


# -------------------------------------------------------------------
# 1. Collect info for all patches
# -------------------------------------------------------------------
patch_files = sorted(PATCH_DIR.glob("*.npy"))
print(f"Found {len(patch_files)} patch files.")

entries = []
pos_dict = {}  # (col,row) -> entry

for p in patch_files:
    patch_idx, col, row, ulx, uly = parse_patch_name(p)

    e = {
        "patch_idx": patch_idx,
        "filename": p.name,
        "rel_path": p.name,
        "col": col,
        "row": row,
        "ulx": ulx,
        "uly": uly,
        "order_id": None,
    }

    key = (col, row)

    # ✅ SAFETY CHECK: ensure spatial uniqueness
    if key in pos_dict:
        raise RuntimeError(
            f"Duplicate (col,row) detected: {key}\n"
            f"  Existing: {pos_dict[key]['filename']}\n"
            f"  New     : {p.name}"
        )

    entries.append(e)
    pos_dict[key] = e


# -------------------------------------------------------------------
# 2. Create spatial ordering (north→south, west→east)
# -------------------------------------------------------------------
all_cols = sorted({e["col"] for e in entries})                 # west → east
all_rows = sorted({e["row"] for e in entries}, reverse=True)   # north → south

order_id = 1
for row in all_rows:
    for col in all_cols:
        key = (col, row)
        if key in pos_dict:
            pos_dict[key]["order_id"] = order_id
            order_id += 1

print(f"Assigned spatial order IDs 1..{order_id - 1}")


# ✅ SAFETY CHECK: ensure every patch received an order_id
missing = [e["filename"] for e in entries if e["order_id"] is None]
if missing:
    raise RuntimeError(
        f"{len(missing)} patches did not receive an order_id.\n"
        f"Examples: {missing[:5]}"
    )


# -------------------------------------------------------------------
# 3. Write metadata CSV
# -------------------------------------------------------------------
out_csv = PATCH_DIR / "patch_metadata_with_order.csv"

with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "order_id",
        "patch_idx",
        "filename",
        "rel_path",
        "col", "row",
        "ulx", "uly",
    ])

    for e in sorted(entries, key=lambda x: x["order_id"]):
        w.writerow([
            e["order_id"],
            e["patch_idx"],
            e["filename"],
            e["rel_path"],
            e["col"],
            e["row"],
            e["ulx"],
            e["uly"],
        ])

print(f"Metadata with spatial order written to: {out_csv}")
